# GCN–Cora: từ đồ thị đến nhãn bài báo
Chạy lần lượt. Cell đầu chuẩn bị repo khi dùng Colab; chạy local từ thư mục repo hoặc notebooks cũng được.

In [ ]:
from pathlib import Path
import os
if not Path('cora.py').exists():
    if Path('../cora.py').exists():
        os.chdir('..')
    else:
        import subprocess
        if not Path('/content/gnn/cora.py').exists():
            subprocess.run(['git', 'clone', 'https://github.com/dungTudonghoa/gnn.git', '/content/gnn'], check=True)
        os.chdir('/content/gnn')
%pip install -q -r requirements.txt

## 1. Tạo đồ thị nhỏ
Node là đối tượng; cạnh là mối quan hệ có hướng. Đây là ví dụ minh họa, chưa phải Cora.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
g = nx.DiGraph([(0,1),(1,2),(1,0),(1,3),(2,3),(3,0)])
nx.draw(g, pos=nx.spring_layout(g, seed=42), with_labels=True, node_color='skyblue', arrows=True)
plt.show()

## 2. Quan sát Cora
Một đồ thị, 2708 bài báo. Mỗi bài có 1433 đặc trưng; nhãn thuộc 7 lớp. `Planetoid` tải tensor đã tiền xử lý, không tải văn bản PDF.

In [ ]:
from cora import load_data, inspect_data
dataset = load_data()
inspect_data(dataset)
data = dataset[0]
print('Shape:', data.x.shape, data.edge_index.shape, data.y.shape)
print('Tổng hàng đầu:', data.x[0].sum().item())

In [ ]:
source, target = data.edge_index[:, 0].tolist()
print('Cạnh đầu:', source, '->', target)
print('Đặc trưng nguồn:', data.x[source])
print('Đặc trưng đích:', data.x[target])
assert not (data.train_mask & data.test_mask).any()
assert not (data.train_mask & data.val_mask).any()
assert not (data.val_mask & data.test_mask).any()

## 3. Huấn luyện
GCN: 1433 → 16 → 7. Chỉ 140 nhãn train được dùng trong loss, nhưng toàn bộ đồ thị tham gia message passing. Test chỉ đánh giá cuối. t-SNE có thể mất thêm thời gian trên CPU.

In [ ]:
from cora import run
model, data, history, metrics = run()

## 4. Quan sát học và biểu diễn
Màu trên hình t-SNE là nhãn thật, không phải nhãn dự đoán.

In [ ]:
from IPython.display import Image, display
plt.plot([row['epoch'] for row in history], [row['loss'] for row in history])
plt.xlabel('Epoch'); plt.ylabel('Train loss'); plt.show()
display(Image(filename='outputs/before.png'))
display(Image(filename='outputs/after.png'))

## 5. Dự đoán một node test
Logits không phải xác suất; softmax đổi sang xác suất trên 7 lớp. Kết quả là một nhãn cho node.

In [ ]:
import torch
model.eval()
with torch.no_grad():
    probabilities = model(data.x, data.edge_index).softmax(dim=1)
node = data.test_mask.nonzero()[0].item()
print('Node:', node)
print('Probabilities:', probabilities[node].cpu().tolist())
print('Predicted:', probabilities[node].argmax().item(), 'True:', data.y[node].item())

## Bài tập
- Vì sao `edge_index` có 2 hàng?
- Vì sao đầu ra có 2708 hàng dù chỉ 140 nhãn train?
- Khác biệt giữa logits, softmax và argmax?
- Vì sao không dùng test accuracy để chọn epoch?

Nguồn và hướng dẫn chi tiết: [README](../README.md).